# EduTech Global · 04 · Market Modeling (Excel Scenario Manager)

**Task 3.** Build Excel-based scenario model projecting Best/Base/Worst-Case revenue for the top-ranked market using documented industry-benchmark price elasticities for B2B SaaS.

**Note on Scenario Manager:** True Scenario Manager is an Excel-native UI feature that openpyxl can't write directly. We instead build a **3-column scenario model** with all assumptions in editable input cells — same effect, openable in any spreadsheet tool, and the user can save scenarios via Excel's Data → What-If Analysis → Scenario Manager menu after opening.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "edutech":
            return parent
        if (parent / "scripts").exists() and (parent / "outputs").exists() and (parent / "data").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Allow either dataset/ or data/ to hold the source CSV; accept both filenames
HR_CSV_NAMES = ["WA_Fn-UseC_-HR-Employee-Attrition.csv", "HR-Employee-Attrition.csv"]
hr_locations = []
for name in HR_CSV_NAMES:
    hr_locations.extend([DATASET_DIR / name, DATA_DIR / name])
HR_CSV_PATH = next((p for p in hr_locations if p.exists()), None)
assert HR_CSV_PATH is not None, (
    f"Could not find HR CSV. Looked for {HR_CSV_NAMES} in {DATASET_DIR} and {DATA_DIR}"
)
print(f"HR data file : {HR_CSV_PATH}")


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference, LineChart
from openpyxl.utils import get_column_letter

# Read the top market from notebook 01
scores = pd.read_csv(DATA_DIR / "attractiveness_scores.csv", index_col=0)
top_market = scores.index[0]
print(f"Top-ranked market for entry: {top_market}")
print(scores)


## Build the workbook

Sheets: **README**, **Assumptions**, **Scenarios** (Best/Base/Worst), **Sensitivity** (price × elasticity grid).

In [ ]:
wb = Workbook()
ARIAL = lambda **kw: Font(name="Arial", **kw)
header_fill = PatternFill("solid", fgColor="1F3864")
input_fill  = PatternFill("solid", fgColor="FFF2CC")
result_fill = PatternFill("solid", fgColor="E2EFDA")
band_fill   = PatternFill("solid", fgColor="F2F2F2")
thin = Side(border_style="thin", color="BFBFBF")
box  = Border(left=thin, right=thin, top=thin, bottom=thin)

# === Sheet 1: README ===
readme = wb.active
readme.title = "README"
readme["A1"] = f"EduTech Global · {top_market} Market Revenue Scenario Model"
readme["A1"].font = ARIAL(size=18, bold=True, color="1F3864")
readme["A2"] = "Task 3 — Best / Base / Worst Case revenue projections"
readme["A2"].font = ARIAL(size=11, italic=True, color="595959")

readme["A4"] = "PURPOSE"; readme["A4"].font = ARIAL(size=12, bold=True)
readme["A5"] = ("Project Best/Base/Worst Case revenue for entering "
                f"the {top_market} B2B SaaS market, varying price points and "
                "applying industry-benchmark price elasticity for enterprise SaaS.")

readme["A7"] = "HOW TO USE"; readme["A7"].font = ARIAL(size=12, bold=True)
steps = [
    "1. Open 'Assumptions' — edit YELLOW cells (price, elasticity, market size, etc.).",
    "2. 'Scenarios' sheet auto-recomputes Best/Base/Worst revenue.",
    "3. 'Sensitivity' shows revenue across a price × elasticity grid.",
    "4. To use Excel's native Scenario Manager: select the input cells on the Assumptions sheet,",
    "   then go to Data → What-If Analysis → Scenario Manager and save your scenarios.",
]
for i, s in enumerate(steps, start=8):
    readme[f"A{i}"] = s

readme["A14"] = "ELASTICITY BENCHMARKS (sources documented on Assumptions sheet)"
readme["A14"].font = ARIAL(size=12, bold=True)
readme["A15"] = "Enterprise B2B SaaS price elasticity typically -0.5 to -1.5"
readme["A16"] = "(Gartner, OpenView Partners SaaS Benchmarks, McKinsey reports)"

readme.column_dimensions["A"].width = 95
print("README sheet built")


In [ ]:
# === Sheet 2: Assumptions ===
asm = wb.create_sheet("Assumptions")
asm["A1"] = f"REVENUE MODEL ASSUMPTIONS · {top_market} Market"
asm["A1"].font = ARIAL(size=16, bold=True, color="FFFFFF")
asm["A1"].fill = header_fill
asm["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
asm.row_dimensions[1].height = 26
asm.merge_cells("A1:D1")

# Market sizing
asm["A3"] = "MARKET SIZING"
asm["A3"].font = ARIAL(size=12, bold=True, color="1F3864")

market_inputs = [
    ("Total addressable market — companies (count)",       150_000, "#,##0"),
    ("Serviceable addressable market (% of TAM)",          0.20,    "0%"),
    ("Year-1 target market share (% of SAM)",              0.10,    "0.0%"),
]
for i, (lbl, val, fmt) in enumerate(market_inputs, start=4):
    asm[f"A{i}"] = lbl; asm[f"B{i}"] = val
    asm[f"A{i}"].font = ARIAL(size=10)
    asm[f"B{i}"].font = ARIAL(size=10, bold=True, color="0000FF")
    asm[f"B{i}"].fill = input_fill; asm[f"B{i}"].border = box
    asm[f"B{i}"].number_format = fmt

# Pricing
asm["A8"] = "PRICING — base/best/worst price points (USD per license per year)"
asm["A8"].font = ARIAL(size=12, bold=True, color="1F3864")

pricing = [
    ("Base case price (USD / license / year)",  300, "$#,##0"),
    ("Best case price (premium tier)",          450, "$#,##0"),
    ("Worst case price (heavy discount)",       180, "$#,##0"),
]
for i, (lbl, val, fmt) in enumerate(pricing, start=9):
    asm[f"A{i}"] = lbl; asm[f"B{i}"] = val
    asm[f"A{i}"].font = ARIAL(size=10)
    asm[f"B{i}"].font = ARIAL(size=10, bold=True, color="0000FF")
    asm[f"B{i}"].fill = input_fill; asm[f"B{i}"].border = box
    asm[f"B{i}"].number_format = fmt

# Elasticity
asm["A13"] = "PRICE ELASTICITY (industry benchmark for B2B SaaS)"
asm["A13"].font = ARIAL(size=12, bold=True, color="1F3864")

elast = [
    ("Base case elasticity",   -1.0, "0.0"),
    ("Best case elasticity",   -0.5, "0.0"),
    ("Worst case elasticity",  -1.5, "0.0"),
]
for i, (lbl, val, fmt) in enumerate(elast, start=14):
    asm[f"A{i}"] = lbl; asm[f"B{i}"] = val
    asm[f"A{i}"].font = ARIAL(size=10)
    asm[f"B{i}"].font = ARIAL(size=10, bold=True, color="0000FF")
    asm[f"B{i}"].fill = input_fill; asm[f"B{i}"].border = box
    asm[f"B{i}"].number_format = fmt

# Sources
asm["A18"] = "SOURCES & DEFINITIONS"
asm["A18"].font = ARIAL(size=12, bold=True, color="1F3864")
notes = [
    "TAM source: Statista B2B SaaS market reports — adjustable",
    "Elasticity source: OpenView SaaS Benchmarks 2023, Gartner research notes",
    "Elasticity formula: % ΔQ = elasticity × % ΔP (relative to base price)",
    "Best case = premium pricing finds buyers (less price-sensitive segment)",
    "Worst case = aggressive discounting still meets resistance (highly elastic demand)",
]
for i, n in enumerate(notes, start=19):
    asm[f"A{i}"] = n
    asm[f"A{i}"].font = ARIAL(size=9, italic=True, color="595959")

asm.column_dimensions["A"].width = 60
asm.column_dimensions["B"].width = 18
asm.column_dimensions["C"].width = 18

print("Assumptions sheet built")


In [ ]:
# === Sheet 3: Scenarios ===
scen = wb.create_sheet("Scenarios")
scen["A1"] = "BEST / BASE / WORST CASE REVENUE PROJECTION"
scen["A1"].font = ARIAL(size=16, bold=True, color="FFFFFF")
scen["A1"].fill = header_fill
scen["A1"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
scen.row_dimensions[1].height = 26
scen.merge_cells("A1:D1")

# Header row
headers = ["Metric", "Worst Case", "Base Case", "Best Case"]
for j, h in enumerate(headers, start=1):
    c = scen.cell(row=3, column=j, value=h)
    c.font = ARIAL(size=11, bold=True, color="FFFFFF")
    c.fill = header_fill; c.border = box
    c.alignment = Alignment(horizontal="center")

# Build rows: each metric, with formula linking to Assumptions sheet
# Layout convention: scenario columns B, C, D = worst, base, best
# Row 4: Price
scen["A4"] = "Price (USD / license / year)"
scen["B4"] = "=Assumptions!B11"  # worst
scen["C4"] = "=Assumptions!B9"   # base
scen["D4"] = "=Assumptions!B10"  # best
for col in "BCD":
    scen[f"{col}4"].number_format = "$#,##0"
    scen[f"{col}4"].font = ARIAL(size=10, color="008000")

# Row 5: Elasticity
scen["A5"] = "Price elasticity"
scen["B5"] = "=Assumptions!B16"  # worst
scen["C5"] = "=Assumptions!B14"  # base
scen["D5"] = "=Assumptions!B15"  # best
for col in "BCD":
    scen[f"{col}5"].number_format = "0.0"
    scen[f"{col}5"].font = ARIAL(size=10, color="008000")

# Row 6: Price delta vs base
scen["A6"] = "% Price change vs base"
scen["B6"] = "=(B4/$C$4)-1"
scen["C6"] = "=(C4/$C$4)-1"
scen["D6"] = "=(D4/$C$4)-1"
for col in "BCD":
    scen[f"{col}6"].number_format = "0.0%"

# Row 7: Quantity change driven by elasticity
scen["A7"] = "% Quantity change (elasticity × Δprice)"
scen["B7"] = "=B5*B6"
scen["C7"] = "=C5*C6"
scen["D7"] = "=D5*D6"
for col in "BCD":
    scen[f"{col}7"].number_format = "0.0%"

# Row 8: Base licenses sold (computed from market sizing on Assumptions)
scen["A8"] = "Base case licenses (from market sizing)"
scen["B8"] = "=Assumptions!B4*Assumptions!B5*Assumptions!B6"
scen["C8"] = "=Assumptions!B4*Assumptions!B5*Assumptions!B6"
scen["D8"] = "=Assumptions!B4*Assumptions!B5*Assumptions!B6"
for col in "BCD":
    scen[f"{col}8"].number_format = "#,##0"

# Row 9: Adjusted licenses sold (base × (1 + qty change))
scen["A9"] = "Adjusted licenses sold"
scen["B9"] = "=B8*(1+B7)"
scen["C9"] = "=C8*(1+C7)"
scen["D9"] = "=D8*(1+D7)"
for col in "BCD":
    scen[f"{col}9"].number_format = "#,##0"

# Row 10: Revenue
scen["A10"] = "REVENUE (USD)"
scen["B10"] = "=B4*B9"
scen["C10"] = "=C4*C9"
scen["D10"] = "=D4*D9"
for col in "BCD":
    scen[f"{col}10"].number_format = "$#,##0"
    scen[f"{col}10"].font = ARIAL(size=12, bold=True, color="00703C")
    scen[f"{col}10"].fill = result_fill

scen["A10"].font = ARIAL(size=12, bold=True)

# Row 11: Revenue vs base
scen["A11"] = "Revenue vs Base Case"
scen["B11"] = "=(B10/$C$10)-1"
scen["C11"] = "=(C10/$C$10)-1"
scen["D11"] = "=(D10/$C$10)-1"
for col in "BCD":
    scen[f"{col}11"].number_format = "+0.0%;-0.0%;0.0%"
    scen[f"{col}11"].font = ARIAL(size=10, bold=True)

# Borders
for r in range(3, 12):
    for c in range(1, 5):
        scen.cell(row=r, column=c).border = box
        if r % 2 == 0 and r > 3:
            if scen.cell(row=r, column=c).fill.start_color.rgb in [None, "00000000"]:
                scen.cell(row=r, column=c).fill = band_fill

# Format the metric column
for r in range(4, 12):
    scen.cell(row=r, column=1).font = ARIAL(size=10, bold=(r in [10, 11]))

# Chart
chart = BarChart()
chart.title = "Revenue by Scenario"
chart.y_axis.title = "Revenue (USD)"
chart.x_axis.title = "Scenario"
data = Reference(scen, min_col=2, min_row=10, max_col=4, max_row=10)
cats = Reference(scen, min_col=2, min_row=3, max_col=4, max_row=3)
chart.add_data(data, titles_from_data=False)
chart.set_categories(cats)
chart.height = 10
chart.width  = 16
scen.add_chart(chart, "A14")

scen.column_dimensions["A"].width = 42
for c in "BCD":
    scen.column_dimensions[c].width = 16

print("Scenarios sheet built with chart")


In [ ]:
# === Sheet 4: Sensitivity (price × elasticity grid) ===
sens = wb.create_sheet("Sensitivity")
sens["A1"] = "SENSITIVITY — Revenue across Price × Elasticity grid"
sens["A1"].font = ARIAL(size=14, bold=True, color="1F3864")
sens.merge_cells("A1:H1")

# Y-axis: prices  | X-axis: elasticities
prices = [180, 220, 260, 300, 340, 380, 420, 460]
elasts = [-0.3, -0.6, -0.9, -1.2, -1.5, -1.8, -2.1]

# Header row (elasticities)
sens["A3"] = "Price (USD) ↓ \ Elasticity →"
sens["A3"].font = ARIAL(size=10, bold=True)
for j, e in enumerate(elasts):
    sens.cell(row=3, column=2+j, value=e).font = ARIAL(size=10, bold=True)
    sens.cell(row=3, column=2+j).number_format = "0.0"

# Grid: revenue = adjusted_qty * price
# adjusted_qty = base_qty * (1 + e * (price/base_price - 1))
# base_qty = TAM * SAM% * Share% (from Assumptions B4*B5*B6)
# base_price = Assumptions!B9
for i, p in enumerate(prices):
    sens.cell(row=4+i, column=1, value=p).number_format = "$#,##0"
    sens.cell(row=4+i, column=1).font = ARIAL(size=10, bold=True)
    for j, e in enumerate(elasts):
        formula = (f"=Assumptions!$B$4*Assumptions!$B$5*Assumptions!$B$6"
                   f"*(1+{e}*({p}/Assumptions!$B$9-1))*{p}")
        c = sens.cell(row=4+i, column=2+j, value=formula)
        c.number_format = "$#,##0"
        c.font = ARIAL(size=9)

# Conditional-formatting style coloring (manual gradient via openpyxl is complex; users can apply CF)
sens["A13"] = "Reading: each cell is total Year-1 revenue (USD) at that price + elasticity combo."
sens["A13"].font = ARIAL(size=9, italic=True, color="595959")
sens.merge_cells("A13:H13")

sens["A14"] = "To apply colour gradient: select the data range → Home → Conditional Formatting → Color Scales."
sens["A14"].font = ARIAL(size=9, italic=True, color="595959")
sens.merge_cells("A14:H14")

sens.column_dimensions["A"].width = 28
for col in range(2, 9):
    sens.column_dimensions[get_column_letter(col)].width = 13

# SAVE
out_file = OUTPUTS_DIR / "Market_Modeling.xlsx"
wb.save(out_file)
print(f"Saved: {out_file}")


## Verify formulas

Let's check the model produces sensible output. (On your machine, opening the file in Excel/LibreOffice will calculate everything.)

In [ ]:
# Quick sanity check — re-implement the math in Python and compare expectations
TAM = 150_000
SAM_PCT = 0.20
SHARE_PCT = 0.10
base_qty = TAM * SAM_PCT * SHARE_PCT
print(f"Base licenses: {base_qty:,.0f}")

scenarios = {
    "Worst" : {"price": 180, "elast": -1.5},
    "Base"  : {"price": 300, "elast": -1.0},
    "Best"  : {"price": 450, "elast": -0.5},
}
base_price = 300

print(f"\n{'Scenario':<8} {'Price':>8} {'Elast':>7} {'%ΔP':>8} {'%ΔQ':>8} {'Qty':>8} {'Revenue':>12}")
for s, p in scenarios.items():
    delta_p = (p["price"] / base_price) - 1
    delta_q = p["elast"] * delta_p
    qty     = base_qty * (1 + delta_q)
    rev     = p["price"] * qty
    print(f"{s:<8} ${p['price']:>7,} {p['elast']:>7.1f} {delta_p:>7.1%} {delta_q:>7.1%} "
          f"{qty:>8,.0f} ${rev:>11,.0f}")

print("\nThe Excel file should produce identical numbers when opened.")


✅ **Notebook 04 complete.** Move to `05_constraint_management_xlsx.ipynb`.